# Python 装饰器完整指南

> 涵盖闭包原理、函数/类装饰器、带参数的装饰器、functools.wraps


## 一、闭包原理（装饰器的基石）

### 1. 什么是闭包？

> **闭包**：函数内部定义函数，内部函数引用了外部函数的变量，即使外部函数执行完毕，这些变量仍被内部函数"记住"。

In [ ]:
def outer(x):           # 外部函数
    def inner(y):       # 内部函数
        return x + y    # 内部函数使用了外部函数的 x
    return inner        # 返回内部函数（不是调用！）

# 创建两个闭包
add_10 = outer(10)      # x = 10 被"封存在" inner 中
add_100 = outer(100)    # x = 100 被"封存在"另一个 inner 中

print(add_10(5))        # 15  (10 + 5)
print(add_100(5))       # 105 (100 + 5)

### 2. 闭包的本质

In [ ]:
def make_multiplier(n):
    def multiplier(x):
        return x * n
    return multiplier

double = make_multiplier(2)
triple = make_multiplier(3)

print(double(5))   # 10
print(triple(5))   # 15

# 查看闭包保存的变量
print(double.__closure__)           # (<cell at ...>,)
print(double.__closure__[0].cell_contents)  # 2

```
闭包 = 函数 + 引用的外部变量环境
      ↓
    就像函数"背着"一个小背包，里面装着它需要的变量
```

## 二、函数装饰器

### 1. 最简单的装饰器

In [ ]:
import functools

def my_decorator(func):         # ① 接收被装饰的函数
    def wrapper():              # ② 定义包装函数
        print("函数执行前...")
        func()                  # ③ 调用原函数
        print("函数执行后...")
    return wrapper              # ④ 返回包装函数

# 使用方式1：手动包装
def say_hello():
    print("Hello!")

say_hello = my_decorator(say_hello)
say_hello()

print("---")

# 使用方式2：@语法糖（推荐）
@my_decorator
def say_goodbye():
    print("Goodbye!")

say_goodbye()

### 2. 带参数和返回值的装饰器

In [ ]:
def logger(func):
    @functools.wraps(func)      # 保留原函数元信息
    def wrapper(*args, **kwargs):
        print(f"调用: {func.__name__}({args}, {kwargs})")
        result = func(*args, **kwargs)
        print(f"返回: {result}")
        return result
    return wrapper

@logger
def add(a, b):
    """计算两数之和"""
    return a + b

result = add(3, 5)
print(f"\n函数名: {add.__name__}")     # add
print(f"文档: {add.__doc__}")           # 计算两数之和

### 3. 为什么需要 `functools.wraps`？

In [ ]:
# ❌ 不用 wraps：原函数信息丢失
def bad_decorator(func):
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@bad_decorator
def my_func():
    """我的函数"""
    pass

print(f"不用 wraps:")
print(f"  名字: {my_func.__name__}")   # wrapper
print(f"  文档: {my_func.__doc__}")    # None

# ✅ 用 wraps：保留原函数信息
def good_decorator(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@good_decorator
def my_func2():
    """我的函数"""
    pass

print(f"\n用 wraps:")
print(f"  名字: {my_func2.__name__}")  # my_func2
print(f"  文档: {my_func2.__doc__}")   # 我的函数

## 三、带参数的装饰器

### 1. 三层嵌套结构

In [ ]:
def repeat(times):              # 第一层：接收装饰器参数
    def decorator(func):        # 第二层：接收被装饰函数
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for _ in range(times):
                result = func(*args, **kwargs)
            return result       # 返回最后一次结果
        return wrapper
    return decorator

# 使用
@repeat(times=3)
def greet(name):
    print(f"Hello, {name}!")

greet("Alice")
# 输出 3 次 Hello, Alice!

### 2. 执行顺序图解

```
@repeat(times=3)
def greet(name): ...

# 等价于：
# 1. repeat(times=3) → 返回 decorator 函数
# 2. decorator(greet) → 返回 wrapper 函数
# 3. greet = wrapper
```

### 3. 实战：权限控制装饰器

In [ ]:
from functools import wraps

def require_role(role):
    """检查用户角色的装饰器"""
    def decorator(func):
        @wraps(func)
        def wrapper(user, *args, **kwargs):
            if user.get("role") != role:
                raise PermissionError(f"需要 {role} 权限")
            return func(user, *args, **kwargs)
        return wrapper
    return decorator

# 使用
@require_role("admin")
def delete_user(user, user_id):
    print(f"{user['name']} 删除了用户 {user_id}")

@require_role("user")
def view_profile(user):
    print(f"查看 {user['name']} 的资料")

admin = {"name": "Alice", "role": "admin"}
normal = {"name": "Bob", "role": "user"}

delete_user(admin, 42)      # ✅ 正常执行
view_profile(normal)         # ✅ 正常执行

# 这会报错
try:
    delete_user(normal, 42)
except PermissionError as e:
    print(f"\n权限错误: {e}")

## 四、类装饰器

### 1. 用类实现装饰器

In [ ]:
class CountCalls:
    """统计函数被调用次数的装饰器类"""

    def __init__(self, func):
        functools.update_wrapper(self, func)  # 类似 @wraps
        self.func = func
        self.count = 0

    def __call__(self, *args, **kwargs):
        self.count += 1
        print(f"{self.func.__name__} 被调用了 {self.count} 次")
        return self.func(*args, **kwargs)

@CountCalls
def say_hello():
    print("Hello!")

say_hello()  # 被调用了 1 次
say_hello()  # 被调用了 2 次
say_hello()  # 被调用了 3 次

### 2. 带参数的类装饰器

In [ ]:
class Timer:
    """计时装饰器（带参数）"""

    def __init__(self, prefix="执行时间"):
        self.prefix = prefix

    def __call__(self, func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            import time
            start = time.time()
            result = func(*args, **kwargs)
            elapsed = time.time() - start
            print(f"[{self.prefix}] {func.__name__}: {elapsed:.4f}s")
            return result
        return wrapper

@Timer(prefix="耗时")
def slow_function():
    import time
    time.sleep(0.5)
    return "done"

slow_function()

## 五、多个装饰器叠加

```
@decorator_a
@decorator_b
@decorator_c
def func():
    pass

# 等价于：func = decorator_a(decorator_b(decorator_c(func)))
# 执行顺序：从下往上包装，从上往下执行
```

In [ ]:
def log_call(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        print(f"[LOG] 调用 {func.__name__}")
        return func(*args, **kwargs)
    return wrapper

def check_auth(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        print(f"[AUTH] 检查权限...")
        return func(*args, **kwargs)
    return wrapper

def cache_result(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        print(f"[CACHE] 检查缓存...")
        return func(*args, **kwargs)
    return wrapper

@log_call
@check_auth
@cache_result
def get_data():
    print("获取数据")

get_data()

## 六、实战案例

### 案例 1：缓存装饰器（简易版 LRU）

In [ ]:
def memoize(func):
    """缓存函数结果"""
    cache = {}

    @functools.wraps(func)
    def wrapper(*args):
        if args not in cache:
            cache[args] = func(*args)
        return cache[args]

    wrapper.cache = cache  # 暴露缓存，方便查看
    return wrapper

@memoize
def fibonacci(n):
    if n < 2:
        return n
    return fibonacci(n - 1) + fibonacci(n - 2)

print(fibonacci(30))   # 瞬间完成
print(f"缓存了 {len(fibonacci.cache)} 个结果")

### 案例 2：重试装饰器

In [ ]:
import time
from functools import wraps

def retry(max_attempts=3, delay=1, exceptions=(Exception,)):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)
                except exceptions as e:
                    print(f"第 {attempt} 次失败: {e}")
                    if attempt == max_attempts:
                        raise
                    time.sleep(delay)
        return wrapper
    return decorator

@retry(max_attempts=3, delay=0.5, exceptions=(ConnectionError,))
def fetch_data(url):
    import random
    if random.random() < 0.7:
        raise ConnectionError("网络错误")
    return f"数据来自 {url}"

# 使用
import random
random.seed(42)
try:
    result = fetch_data("https://api.example.com")
    print(result)
except ConnectionError:
    print("最终失败")

### 案例 3：属性验证装饰器

In [ ]:
def validate(**validators):
    """验证函数参数"""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(**kwargs):
            for key, validator in validators.items():
                if key not in kwargs:
                    raise ValueError(f"缺少参数: {key}")
                if not validator(kwargs[key]):
                    raise ValueError(f"参数 {key} 验证失败")
            return func(**kwargs)
        return wrapper
    return decorator

@validate(
    age=lambda x: isinstance(x, int) and x > 0,
    name=lambda x: isinstance(x, str) and len(x) > 0
)
def create_user(name, age):
    print(f"创建用户: {name}, {age}岁")

create_user(name="Alice", age=25)   # ✅

# 这会报错
try:
    create_user(name="", age=25)
except ValueError as e:
    print(f"验证失败: {e}")

## 七、知识图谱总结

```
装饰器
├── 基础：闭包
│   ├── 函数嵌套函数
│   ├── 内部函数引用外部变量
│   └── 外部函数返回内部函数
│
├── 函数装饰器
│   ├── 无参数：@decorator
│   │   └── def wrapper(*args, **kwargs)
│   ├── 带参数：@decorator(args)
│   │   └── 三层嵌套
│   └── @functools.wraps(func)
│       └── 保留 __name__, __doc__ 等元信息
│
├── 类装饰器
│   ├── __init__(self, func)  或 __init__(self, *args)
│   └── __call__(self, *args, **kwargs)
│
├── 多个装饰器叠加
│   └── 从下往上包装，从上往下执行
│
└── 实战应用
    ├── 日志记录
    ├── 权限验证
    ├── 缓存 memoize
    ├── 重试机制 retry
    ├── 参数验证
    └── 性能计时
```

### 关键要点速查

| 主题 | 核心要点 |
|------|---------|
| **闭包** | 函数 + 外部变量环境，外部函数返回内部函数 |
| **@语法糖** | `@decorator` 等价于 `func = decorator(func)` |
| **wraps** | 必须用，否则原函数元信息丢失 |
| **带参数装饰器** | 三层嵌套：参数 → 接收函数 → 包装执行 |
| **类装饰器** | 实现 `__init__` 和 `__call__` |
| **叠加顺序** | 从下往上包装，调用时从上往下执行 |